# Auto Insurance Fraud Analysis
Week 1 Friday Project 
## Business Context
Analysis of 1,000 auto insurance claims across Ohio, Indiana, 
and Illinois to identify fraud predictors and high-rish claims types. 


In [31]:
# loading data
import pandas as pd 

df = pd.read_csv('insurance_claims.csv')

print(df.head())
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

   months_as_customer  age  policy_number policy_bind_date policy_state  \
0                 328   48         521585       2014-10-17           OH   
1                 228   42         342868       2006-06-27           IN   
2                 134   29         687698       2000-09-06           OH   
3                 256   41         227811       1990-05-25           IL   
4                 228   44         367455       2014-06-06           IL   

  policy_csl  policy_deductable  policy_annual_premium  umbrella_limit  \
0    250/500               1000                1406.91               0   
1    250/500               2000                1197.22         5000000   
2    100/300               2000                1413.14         5000000   
3    250/500               2000                1415.74         6000000   
4   500/1000               1000                1583.91         6000000   

   insured_zip  ... police_report_available total_claim_amount injury_claim  \
0       466132  ...      

In [42]:
# data cleaning 

#dropping empty column

df = df.drop(columns=['_c39'])

#checking fraud distribution
print(df['fraud_reported'].value_counts())
print()

#checking how many stasted
print(df['policy_state'].value_counts())

fraud_reported
N    753
Y    247
Name: count, dtype: int64

policy_state
OH    352
IL    338
IN    310
Name: count, dtype: int64


In [33]:
# Fraud distribution overview
print("Overall fraud rate:")
print(df['fraud_reported'].value_counts())
print()
print(df['fraud_reported'].value_counts(normalize=True).round(3) * 100)

Overall fraud rate:
fraud_reported
N    753
Y    247
Name: count, dtype: int64

fraud_reported
N    75.3
Y    24.7
Name: proportion, dtype: float64


In [34]:
# total claim amount by state 
print(df.groupby('policy_state')['total_claim_amount'].sum().map('{:,.2f}'.format))

policy_state
IL    17,861,330.00
IN    16,432,160.00
OH    18,468,450.00
Name: total_claim_amount, dtype: object


In [35]:
# share of total claims per state 
total = df['total_claim_amount'].sum()
result = df.groupby('policy_state')['total_claim_amount'].sum()
pct = (result / total * 100).round(1)
print(pct)

policy_state
IL    33.9
IN    31.1
OH    35.0
Name: total_claim_amount, dtype: float64


In [41]:
# Claim amount by age and sex
df['age_category'] = pd.cut(df['age'], 
  bins=[16, 25, 40, 65, 100],
  labels=['Teen/Young', 'Young Adult', 'Middle age', 'Senior'])
print(df['age_category'].value_counts())

result = df.groupby(['age_category', 'insured_sex'], 
                    observed=True)['total_claim_amount'].mean().round(2).reset_index()
result.columns = ['Age Category', 'Sex', 'Avg Claim']
print(result)


age_category
Young Adult    553
Middle age     407
Teen/Young      40
Senior           0
Name: count, dtype: int64
  Age Category     Sex  Avg Claim
0   Teen/Young  FEMALE   50876.55
1   Teen/Young    MALE   61772.73
2  Young Adult  FEMALE   51835.21
3  Young Adult    MALE   48894.88
4   Middle age  FEMALE   56087.26
5   Middle age    MALE   55259.62


In [40]:
#Fraud by state 
result = df.groupby('policy_state')['fraud_reported'].value_counts(normalize=True).round(3) *100
print(result)

policy_state  fraud_reported
IL            N                 77.2
              Y                 22.8
IN            N                 74.5
              Y                 25.5
OH            N                 74.1
              Y                 25.9
Name: proportion, dtype: float64


In [37]:
# Fraud by incident type

result = df.groupby('incident_type')['fraud_reported'].value_counts(normalize=True).round(3) *100
print(result)

incident_type             fraud_reported
Multi-vehicle Collision   N                 72.8
                          Y                 27.2
Parked Car                N                 90.5
                          Y                  9.5
Single Vehicle Collision  N                 71.0
                          Y                 29.0
Vehicle Theft             N                 91.5
                          Y                  8.5
Name: proportion, dtype: float64


In [38]:
# Fraud by collision type 

result = df.groupby('collision_type')['fraud_reported'].value_counts(
    normalize=True).round(3) * 100
print(result)

collision_type   fraud_reported
?                N                 91.0
                 Y                  9.0
Front Collision  N                 72.4
                 Y                 27.6
Rear Collision   N                 68.8
                 Y                 31.2
Side Collision   N                 74.6
                 Y                 25.4
Name: proportion, dtype: float64


## Key findings and Recommendation 
single vehicle collision claims have the highest fraud rate at 29%, which can be explained by the absence of independent witnesses, making these incidents easier to fabricate. 
Rear collision claims have the highest fraud rate within collision types at 31.2%, likely because they are easily staged. 
I would recommend the company request specific documentation for these claim types such as dashcam footage and mandatory authority investigations. As well, apply more rigorous scrutiny before approving payouts.
